In [1]:
import os
import time
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI


In [2]:
# Load environment variables from current directory .env and override existing ones
load_dotenv(".env", override=True)

# load api key from environment
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY not found in .env file")

client = OpenAI(api_key=api_key)
MAX_RETRIES = 6
MODEL_ID = "gpt-4.1-2025-04-14"



In [3]:
# Path to the summarized principles JSON

principles_path = Path(r"C:\Users\laras\OneDrive\Dokumente\duke_classes\icai_project\Principle-Elicitation\DecontextualPrinciples\self_written_guiding_principles_summarized.json")

evaluator_prompt_path = Path("judge_few_shot_examples_dimensions.txt")

with principles_path.open("r", encoding="utf-8") as f:
    principles_raw = json.load(f)

if isinstance(principles_raw, list):
    principles_data = {str(i): item for i, item in enumerate(principles_raw)}
else:
    principles_data = principles_raw

with evaluator_prompt_path.open("r", encoding="utf-8") as f:
    evaluator_prompt = f.read()

print("Loaded principles from:", principles_path)
print("Loaded evaluator prompt from:", evaluator_prompt_path)



Loaded principles from: C:\Users\laras\OneDrive\Dokumente\duke_classes\icai_project\Principle-Elicitation\DecontextualPrinciples\self_written_guiding_principles_summarized.json
Loaded evaluator prompt from: judge_few_shot_examples_dimensions.txt


In [4]:
evaluator_prompt

'# Identity\n\nYou are an impartial evaluator that scores how well principles conform to seven evaluative dimensions.\n\nInstructions\nRate each principle based on the dimensions listed below.\nProvide a separate Likert scale rating that indicates how well the principle agrees with each dimension, based on the following scale:\n\n1 (Strongly disagree)\n2 (Disagree)\n3 (Neither agree nor disagree)\n4 (Agree)\n5 (Strongly agree)\n\nIn your response, only specify the dimension name and numerical rating. Do not elaborate or justify the ratings.\n\n# Dimensions\n\n[Moral Grounding] The principle is informed by Moral Grounding.\n\n[Fairness] The principle treats all affected parties equally and without bias.\n\n[Generality] The principle is broadly applicable across different contexts.\n\n[Clarity] The wording and intent of the principle are interpretable and unambiguous.\n\n[Applicability] The principle can feasibly be actioned upon and be achieved by the AI in a real-world context.\n\n[Con

In [5]:
def parse_dimension_scores(raw_text: str):
    scores = {}
    for line in raw_text.strip().splitlines():
        m = re.match(r"\s*\[(.*?)\]\s*(?:--|:)\s*(\d+)", line)
        if m:
            dim_name = m.group(1).strip()
            score = int(m.group(2))
            scores[dim_name] = score
    return scores



In [6]:
results = []
total = len(principles_data)
processed = 0

for key, item in principles_data.items():
    cluster_id = item.get("cluster_id", key)
    summarized_principle = item.get("summarized_principle", "").strip()

    if not summarized_principle:
        continue

    user_content = (
        "Principle evaluated:\n"
        f"{summarized_principle}\n\n"
        "Evaluator output:"
    )

    retries = 0
    while True:
        try:
            resp = client.chat.completions.create(
                model=MODEL_ID,
                messages=[
                    {"role": "system", "content": evaluator_prompt},
                    {"role": "user", "content": user_content},
                ],
                max_tokens=256,
                temperature=0,
            )
            judge_text = resp.choices[0].message.content
            break
        except Exception as e:
            retries += 1
            if retries > MAX_RETRIES:
                raise
            print(f"Error ({e}), retry {retries}...")
            time.sleep(min(30, 2 ** retries))

    scores = parse_dimension_scores(judge_text)
    results.append(
        {
            "cluster_id": cluster_id,
            "summarized_principle": summarized_principle,
            "raw_judge_output": judge_text,
            "scores": scores,
        }
    )

    processed += 1

    if processed % 50 == 0:
        print(f"Scored {processed} / {total} principles so far...")

print(f"Evaluated {len(results)} principles (out of {total}).")



Scored 50 / 496 principles so far...
Scored 100 / 496 principles so far...
Scored 150 / 496 principles so far...
Scored 200 / 496 principles so far...
Scored 250 / 496 principles so far...
Scored 300 / 496 principles so far...
Scored 350 / 496 principles so far...
Scored 400 / 496 principles so far...
Scored 450 / 496 principles so far...
Evaluated 496 principles (out of 496).


In [7]:
rows = []
for r in results:
    base = {
        "cluster_id": r["cluster_id"],
        "summarized_principle": r["summarized_principle"],
    }
    for dim, score in r["scores"].items():
        base[dim] = score
    rows.append(base)

df = pd.DataFrame(rows)

non_dim_cols = {"cluster_id", "summarized_principle"}
score_cols = [c for c in df.columns if c not in non_dim_cols]

df["average_score"] = df[score_cols].mean(axis=1)

for r, avg in zip(results, df["average_score"]):
    r["average_score"] = float(avg)

output_json_path = Path("principle_dimension_scores_self_written_guiding_principles.json")
with output_json_path.open("w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("Saved results (including average_score) to:", output_json_path)


Saved results (including average_score) to: principle_dimension_scores_self_written_guiding_principles.json


In [8]:
print("\nTop 20 principles by average_score (guiding life principles):")
top15 = df.sort_values("average_score", ascending=False).head(20)
for _, row in top15.iterrows():
    print(f"[avg={row['average_score']:.2f}] (cluster_id={row['cluster_id']}) {row['summarized_principle']}")




Top 20 principles by average_score (guiding life principles):
[avg=4.86] (cluster_id=22) **Summarized Principle:**  
A core value is lifelong learning, curiosity, and openness to new perspectives. This includes being observant, learning from every experience and person, seeking knowledge continuously beyond formal education, and sharing what is learned with others. It also means encouraging others—especially children—to explore diverse activities, cultures, and passions to broaden their horizons and maintain an open, inquisitive mindset throughout life.
[avg=4.86] (cluster_id=54) **Summarized Principle:**  
Empathy and compassion are essential guiding values in life. It is important to strive to understand others' perspectives, practice kindness, and remain open-minded, even toward those you may disagree with. By putting yourself in other people's shoes and recognizing how different experiences shape people's actions and behavior, you can help create more positive, understanding, and 

In [10]:
print("\nBottom 20 principles by average_score (guiding life principles):")
bottom15 = df.sort_values("average_score", ascending=True).head(20)
for _, row in bottom15.iterrows():
    print(f"[avg={row['average_score']:.2f}] (cluster_id={row['cluster_id']}) {row['summarized_principle']}")


Bottom 20 principles by average_score (guiding life principles):
[avg=1.00] (cluster_id=159) The summarized principle is:

- No core principles or values expressed (i.e., none provided or stated).
[avg=2.00] (cluster_id=400) Summarized Principle:  
I value friendships with people who are open about their feelings; I haven't considered having children.
[avg=2.14] (cluster_id=483) A world without religion would be better; hard work should determine outcomes (with some luck), people should value silence, and society is becoming less intelligent.
[avg=2.29] (cluster_id=268) Treat everyone with respect and kindness regardless of identity, stay true to yourself, live authentically for yourself (not just for others), hold left-leaning values, and oppose AI-generated art.
[avg=2.43] (cluster_id=481) **Summarized Principle:**  
A pessimistic view that life is full of hardships, plans often fail, and ideals like the "American dream" are just consumerist illusions, leading to regret about existe